# Stochastic Calculus, Step by Step · Lesson 28: The Feynman-Kac theorem

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. An expectation that depends on where you start · 3. The theorem · 4. Adding a discount · 5. Back to Black-Scholes · 6. A rate that moves: bonds under Vasicek · 7. Exercises · 8. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 28,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. An expectation that depends on where you start

Let $dX = a(X)\,dt + b(X)\,dW$, and define $g(t, x) = E[h(X_T) \mid X_t = x]$. Because the future of $X$ depends only on its present value (it is a **Markov** process), $g$ is an ordinary function of two numbers. Take $X = W$ and $h(x) = x^2$, where the answer is known: $g(t, x) = x^2 + (T - t)$.

In [ ]:
rng = np.random.default_rng(seed)
M_ = 1_000_000
rows = []
Zv = rng.standard_normal(M_)
for t_, x_ in ((0.0, 0.0), (0.0, 1.0), (0.5, 1.0), (0.5, -2.0), (0.9, 0.5)):
    X_T = x_ + math.sqrt(T - t_) * Zv
    rows.append({"t": t_, "x": x_, "mc": float((X_T**2).mean()), "exact": x_**2 + (T - t_)})
out["start"] = {"rows": rows}
pd.DataFrame(rows)

## 3. The theorem

$g(t, X_t)$ is a martingale, by the tower property of Lesson 8. Itô's lemma says its $dt$ term is $g_t + ag_x + \tfrac12b^2g_{xx}$, and a martingale's $dt$ term is zero. So

$$g_t + a\,g_x + \tfrac12b^2g_{xx} = 0, \qquad g(T, x) = h(x) .$$

Every expectation of this kind solves a PDE. Checked for three cases by bumping the known answer:

In [ ]:
def fk_residual(g, a_, b_, t_, x_, h=1e-3, k=1e-4):
    gt = (g(t_ + k, x_) - g(t_ - k, x_)) / (2 * k)
    gx = (g(t_, x_ + h) - g(t_, x_ - h)) / (2 * h)
    gxx = (g(t_, x_ + h) - 2 * g(t_, x_) + g(t_, x_ - h)) / h**2
    return gt + a_(x_) * gx + 0.5 * b_(x_) ** 2 * gxx


cases = [("W, payoff x^2", lambda t_, x_: x_**2 + (T - t_), lambda x_: 0.0, lambda x_: 1.0),
         ("W, payoff e^x", lambda t_, x_: math.exp(x_ + 0.5 * (T - t_)), lambda x_: 0.0, lambda x_: 1.0),
         ("GBM at 8%, payoff S", lambda t_, x_: x_ * math.exp(mu * (T - t_)), lambda x_: mu * x_, lambda x_: sigma * x_)]
res_rows = [{"case": c[0], "resid": max(abs(fk_residual(c[1], c[2], c[3], t_, x_)) for t_ in (0.2, 0.5, 0.8) for x_ in (0.5, 1.0, 1.5))} for c in cases]
# the martingale, along simulated paths: the running value of g(t, W_t) has a constant average
gen = np.random.default_rng(seed + 1)
Wm = gen.normal(0, math.sqrt(0.25), (400_000, 4)).cumsum(axis=1)
tm = np.array([0.25, 0.5, 0.75, 1.0])
g_run = Wm**2 + (T - tm)
out["thm"] = {"rows": res_rows, "means": [{"t": float(tm[i]), "mean": float(g_run[:, i].mean()), "sd": float(g_run[:, i].std())} for i in range(4)]}
pd.DataFrame(res_rows)

## 4. Adding a discount

With a constant discount rate $r$, take $g(t, x) = E[e^{-r(T - t)}h(X_T) \mid X_t = x]$. Now $e^{-rt}g(t, X_t)$ is the martingale, and the extra factor brings in one more term:

$$g_t + a\,g_x + \tfrac12b^2g_{xx} = rg .$$

In [ ]:
# the discounted version, for GBM at rate r with payoff S^2: the Lesson 19 answer
gd = lambda t_, x_: x_**2 * math.exp((r + sigma**2) * (T - t_))
disc_rows = []
for t_, x_ in ((0.0, 100.0), (0.5, 100.0), (0.5, 120.0)):
    tau = T - t_
    S_ = x_ * np.exp((r - 0.5 * sigma**2) * tau + sigma * math.sqrt(tau) * Zv)
    disc_rows.append({"t": t_, "x": x_, "mc": float(math.exp(-r * tau) * (S_**2).mean()), "exact": gd(t_, x_)})
out["disc"] = {"rows": disc_rows, "resid": abs(fk_residual(gd, lambda x_: r * x_, lambda x_: sigma * x_, 0.5, 100.0, h=0.01, k=1e-4) - r * gd(0.5, 100.0)) }
pd.DataFrame(disc_rows)

## 5. Back to Black-Scholes

With $a = rS$, $b = \sigma S$, the discounted equation **is** the Black-Scholes equation of Lesson 19, and the expectation **is** the risk-neutral price of Lesson 26. Two different roads reach the same object.

Which gives an answer to the puzzle of $\mu$. The equation has $r$ in the $S$-term because the expectation is under $\tilde P$. Compute a discounted expectation under the real probability instead, and you get a different function that solves a different equation, with $\mu$ where $r$ was. It satisfies its own PDE, but it is not a price: no portfolio replicates it for that money.

In [ ]:
def expect_call(s, tau, drift, disc_rate=r):
    # e^{-disc_rate tau} E[(S_T - K)+] when S has drift `drift`; closed form.
    dd2 = (math.log(s / K) + (drift - 0.5 * sigma**2) * tau) / (sigma * math.sqrt(tau))
    dd1 = dd2 + sigma * math.sqrt(tau)
    return math.exp(-disc_rate * tau) * (s * math.exp(drift * tau) * N(dd1) - K * N(dd2))


ec = lambda t_, x_: expect_call(x_, T - t_, mu)
ec_r = lambda t_, x_: expect_call(x_, T - t_, r)
pde_with = lambda g_, drift, t_, x_: fk_residual(g_, lambda s: drift * s, lambda s: sigma * s, t_, x_, h=0.01) - r * g_(t_, x_)
out["puzzle"] = {"rn_value": ec_r(0.0, S0), "real_value": ec(0.0, S0), "bs": float(call(S0, T)),
                 "rn_in_r": abs(pde_with(ec_r, r, 0.5, S0)), "real_in_mu": abs(pde_with(ec, mu, 0.5, S0)), "real_in_r": abs(pde_with(ec, r, 0.5, S0)),
                 "extra_term": (mu - r) * S0 * float((ec(0.5, S0 + 0.01) - ec(0.5, S0 - 0.01)) / 0.02)}
out["puzzle"]

## 6. A rate that moves: bonds under Vasicek

So far the discount rate has been constant. Let it follow $dr_t = \kappa(\bar r - r_t)\,dt + s\,dW_t$ (the **Vasicek** model). The bond price $P(t, r) = \tilde E\big[e^{-\int_t^Tr_u\,du} \mid r_t = r\big]$ has a discount that moves with the state, and Feynman-Kac still works: the $rg$ on the right becomes $r\cdot g$, with $r$ now a variable.

$$P_t + \kappa(\bar r - r)P_r + \tfrac12s^2P_{rr} = rP, \qquad P(T, r) = 1 .$$

The solution has the form $P = e^{A(\tau) - B(\tau)r}$, with $B(\tau) = (1 - e^{-\kappa\tau})/\kappa$. Against Monte Carlo:

In [ ]:
kappa, r_bar, s_v, r_0 = 0.5, 0.04, 0.01, 0.03


def vasicek_bond(r_now, tau):
    B_ = (1 - math.exp(-kappa * tau)) / kappa
    A_ = (r_bar - s_v**2 / (2 * kappa**2)) * (B_ - tau) - s_v**2 * B_**2 / (4 * kappa)
    return math.exp(A_ - B_ * r_now), A_, B_


gen = np.random.default_rng(seed + 2)
v_rows = []
for tau in (1.0, 2.0, 5.0, 10.0):
    n_t = int(round(tau * 100)); dt_ = tau / n_t
    rr = np.full(100_000, r_0); integ = np.zeros(100_000)
    for _ in range(n_t):
        rr_new = rr + kappa * (r_bar - rr) * dt_ + s_v * math.sqrt(dt_) * gen.standard_normal(100_000)
        integ += 0.5 * (rr + rr_new) * dt_
        rr = rr_new
    d_ = np.exp(-integ)
    p_, a_, b_ = vasicek_bond(r_0, tau)
    v_rows.append({"tau": tau, "A": a_, "B": b_, "formula": p_, "mc": float(d_.mean()), "se": float(d_.std() / math.sqrt(d_.size)), "flat": math.exp(-r_0 * tau),
                   "yld": -math.log(p_) / tau * 100})
h_ = 1e-4
gv = lambda t_, x_: vasicek_bond(x_, 5.0 - t_)[0]
vr_resid = fk_residual(gv, lambda x_: kappa * (r_bar - x_), lambda x_: s_v, 1.0, r_0, h=1e-3, k=1e-4) - r_0 * gv(1.0, r_0)
out["vasicek"] = {"rows": v_rows, "kappa": kappa, "r_bar": r_bar, "s": s_v, "r0": r_0, "resid": abs(vr_resid)}
pd.DataFrame(v_rows)

## 7. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 4: Solutions*.

**Exercise 1.** Show that $g(t, X_t) = E[h(X_T) \mid \mathcal F_t]$ is a martingale, using the tower property. Where does the Markov property come in, in saying that this conditional expectation is a function of $X_t$ alone?

**Exercise 2.** Using Feynman-Kac with $X = W$, write down the equation and terminal condition for $h(x) = x^2$ and for $h(x) = e^x$. Solve each by guessing a form, $x^2 + c(t)$ and $e^{x + c(t)}$, and check the answers against direct expectations.

**Exercise 3.** Prove the discounted version: show that $e^{-rt}g(t, X_t)$ is a martingale, apply Itô's lemma, and set the $dt$ term to zero.

**Exercise 4.** Let $S$ be a GBM with drift $\mu$ and $h(S) = S$. Write the equation for $g = E[S_T \mid S_t = S]$, guess $g = Sc(t)$, and solve. Then do the same with the discount, and show the discounted price is $S$ when $\mu = r$.

**Exercise 5.** A friend says: "Feynman-Kac shows the option's value is an expectation under the real probability, with $\mu$." Explain precisely what is wrong, by writing out which expectation solves which equation. What is the extra term that a real-world expectation leaves in the Black-Scholes equation?

**Exercise 6.** For Vasicek, substitute $P = e^{A(\tau) - B(\tau)r}$ into the bond equation, with $\tau = T - t$. Collect the terms with $r$ and without. Solve for $B$, and write down the equation for $A$ and its solution.

**Exercise 7.** (Code) Use Feynman-Kac with $X = W$ and $h(x) = x^4$: guess the polynomial $g = x^4 + c_1(t)x^2 + c_2(t)$, solve for $c_1$ and $c_2$ by hand, and confirm by simulation at several $(t, x)$ and by computing the PDE residual with bumps.

**Exercise 8.** (Code) Price Vasicek bonds by Monte Carlo at maturities 1, 2, 5, 10 with the lesson's parameters. Compare the formula, and find how the error depends on the number of time steps per year (10, 50, 250). Use both a left-endpoint sum and a trapezium sum for $\int r\,dt$.

## 8. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")